# Data Audit Sandbox
## Decision Science · Chapter 2 Extension · English
**Core activity: 25 minutes.** Open this `.ipynb` at https://colab.research.google.com using **File → Upload notebook**, connect to a Python runtime, then run cells in order. No API key, paid AutoML service, or separate data upload is required. Colab supplies the standard packages used below. The main activity needs no internet downloads after opening.

**Decision:** Should a fictional lender use its historical model for all applicants next quarter, including first-time customers?

**Prediction time:** the application, before approval. **Target:** default within 12 months if the applicant receives the specified loan. All historical loans in this simulation have completed their 12-month follow-up. Loan terms are held fixed.

The loan data are **synthetic**, created for this class. Simulated potential outcomes for rejected applicants are a teaching device, never real observed repayment records. The optional final activity uses a real UCI dataset, embedded with attribution.

This notebook uses **rule-based profiling and automated KNN model selection**. It does not call a generative AI or commercial AutoML service. Automation follows the rules we give it; the audit asks whether those rules serve the decision.

**Work in pairs:** one operator, one skeptical reviewer. Stop at each discussion prompt before continuing. Record exact numbers and distinguish observation, inference, and unknown.


## 0 · Setup
Run this cell. If a local Jupyter installation lacks packages, install `numpy pandas scikit-learn matplotlib` in that environment first. The version printout belongs in your submission.


In [ ]:
import sys, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score
from IPython.display import display
print({"Python":platform.python_version(), "numpy":np.__version__, "pandas":pd.__version__, "sklearn":sklearn.__version__})


## 1 · Controls and data contract
Run the default first. Later change **one** value and use Runtime → Run all. A new seed is a sensitivity check, not a way to hunt for a better score.

| Field | Meaning and availability |
|---|---|
| applicant_id | Unique application identifier; audit only, excluded from predictors |
| segment | established = existing customer; new = first-time customer; known at application |
| income_k | Annual income in thousands of fictional currency units; positive by data contract |
| credit_score | Fictional application score between 300 and 850 |
| debt_ratio | Monthly debt payments / monthly income, between 0 and 1 |
| approved | Historical policy decision; determines inclusion, excluded from predictors |
| default_12m | Recorded 12-month default, 1 = default; only observed for approved loans |
| collection_flag | Collection record created after the outcome window; deliberately leaks recorded target |

The generator also holds `oracle_default_if_funded` in a separate answer vault. It is excluded from profiling and all model fitting. In real data this counterfactual is usually unavailable for rejected applicants.


In [ ]:
# @title Classroom controls
SEED = 42 # @param {type:"integer"}
APPROVAL_CUTOFF = 640 # @param [620, 640, 680] {type:"raw"}
FUTURE_NEW_SHARE = 0.45 # @param [0.2, 0.45, 0.7] {type:"raw"}
LABEL_MISS_RATE = 0.20 # @param [0.0, 0.2, 0.4] {type:"raw"}
K_VALUES = [1, 5, 25, 100]
assert 0 <= LABEL_MISS_RATE < 1 and 0 < FUTURE_NEW_SHARE < 1

def generate(n, new_share, seed, offset):
    r = np.random.default_rng(seed)
    new = r.random(n) < new_share
    income = np.exp(r.normal(4.0 - 0.18*new, 0.42, n))
    credit = np.clip(r.normal(680 - 40*new, 55, n), 300, 850)
    debt = np.clip(r.beta(2.3, 4.5, n) + 0.10*new, 0, 1)
    # Teaching mechanism (hidden from the model): for first-time customers the observed fields carry less signal
    # (thin files), and they also carry extra risk that no column records. Established customers behave as history suggests.
    signal = 14*debt + 3.5*(650-credit)/65 - 1.2*np.log(income/55)
    risk = 1/(1+np.exp(-(-5.8 + (1-0.85*new)*signal + 4.5*new)))
    potential = r.binomial(1, risk)
    recorded = potential.copy()
    recorded[(potential == 1) & (r.random(n) < LABEL_MISS_RATE)] = 0
    approved = (~new) & (credit >= APPROVAL_CUTOFF)
    d = pd.DataFrame({'applicant_id':np.arange(offset, offset+n),
        'segment':np.where(new,'new','established'), 'income_k':income,
        'credit_score':credit, 'debt_ratio':debt, 'approved':approved,
        'default_12m':np.where(approved,recorded,np.nan),
        'collection_flag':np.where(approved,recorded,np.nan)})
    # Inject data-entry issues separately from the approval policy.
    d.loc[r.random(n) < np.where(new,0.30,0.08),'income_k'] = np.nan
    bad = r.choice(n, int(n*0.01), replace=False)
    d.loc[bad,'income_k'] = -999
    return d, pd.Series(potential,index=d.applicant_id,name='oracle_default_if_funded')

applications, historical_vault = generate(6000,0.30,SEED,0)
future, future_vault = generate(3000,FUTURE_NEW_SHARE,SEED+1000,100000)
# Future application-time records have no observed repayment outcomes yet.
future[['default_12m','collection_flag']] = np.nan
future = future.drop(columns='approved')
history = applications.loc[applications.approved].copy()
raw = pd.concat([history,history.sample(40,random_state=SEED)],ignore_index=True)
print('Historical applicant register:',len(applications),'| Recorded loan rows:',len(raw),'| Future applicants:',len(future))
display(raw.head())


## 2 · Automated technical report (4 minutes)
Run the audit. Which findings are facts about the file? Which decisions require the data contract?

**Before cleaning:** Would you delete every incomplete record? What happens to the first-time-customer share if income is missing more often in that group?


In [ ]:
def profile(d):
    return pd.DataFrame({'dtype':d.dtypes.astype(str),'missing_n':d.isna().sum(),
                         'missing_pct':100*d.isna().mean(),'unique_n':d.nunique()})
display(profile(raw))
print('Exact duplicate rows:',int(raw.duplicated().sum()))
print('Non-positive income entries:',int((raw.income_k <= 0).sum()))
print('Recorded default balance:'); display(raw.default_12m.value_counts(normalize=True).rename('share'))
display(raw[['income_k','credit_score','debt_ratio']].describe())
print('Future missing income by segment, BEFORE any row deletion:')
display(future.assign(missing=future.income_k.isna()).groupby('segment').missing.agg(['size','mean']))
print('New-customer share before versus after dropping missing-income rows:',
      round(future.segment.eq('new').mean(),3),
      round(future.dropna(subset=['income_k']).segment.eq('new').mean(),3))


## 3 · Documented cleaning, then separate data (3 minutes)
The import log tells us the 40 exact duplicates are accidental copies of the **same application**, so remove them before splitting. Repeated real events would need different handling. The data contract makes negative annual income invalid here, so turn it into missing. Keep valid extreme incomes.

Split unique applications into **60% training, 20% validation, 20% test**, stratified by the recorded label. Fit imputation and scaling on training only. These deterministic schema checks do not learn from test data. Production documentation, including feature timing, should also be reviewed before fitting.

**Median imputation** fills a blank with the middle training value. It makes the software run; it does not recover the applicant's true income.


In [ ]:
def schema_clean(d):
    d=d.copy()
    d.loc[d.income_k <= 0,'income_k']=np.nan
    return d
clean=schema_clean(raw.drop_duplicates())
assert clean.applicant_id.is_unique
train, remaining=train_test_split(clean,test_size=0.40,random_state=SEED,stratify=clean.default_12m)
valid, test=train_test_split(remaining,test_size=0.50,random_state=SEED,stratify=remaining.default_12m)
assert set(train.applicant_id).isdisjoint(test.applicant_id)
assert set(valid.applicant_id).isdisjoint(test.applicant_id)
FEATURES=['income_k','credit_score','debt_ratio']
def knn(k):
    return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
print('Rows after duplicate removal:',len(clean))
print('Training / validation / test:',len(train),len(valid),len(test))
print('Income blanks after schema check:',int(clean.income_k.isna().sum()))
print('Eligible features:',FEATURES)


## 4A · Which K should we choose? (3 minutes)
KNN predicts from similar applicants. **Small K is more flexible**; large K smooths across more neighbors. Scaling prevents currency amounts from dominating the distance just because of their units.

Automation tries four K values and chooses the highest **validation balanced accuracy**. Ties favor larger K. The test set does not choose the model.

**Predict before running:** Which K fits the training data most closely? After the run: which K should we choose, and why is it not the same one?


In [ ]:
rows=[]; models={}
for k in K_VALUES:
    m=knn(k).fit(train[FEATURES],train.default_12m)
    models[k]=m
    rows.append({'K':k,'train_error':1-accuracy_score(train.default_12m,m.predict(train[FEATURES])),
        'validation_error':1-accuracy_score(valid.default_12m,m.predict(valid[FEATURES])),
        'validation_balanced_accuracy':balanced_accuracy_score(valid.default_12m,m.predict(valid[FEATURES]))})
search=pd.DataFrame(rows)
best_k=int(search.sort_values(['validation_balanced_accuracy','K'],ascending=[False,False]).iloc[0].K)
best=models[best_k]
display(search.round(3)); print('Selected K:',best_k)
ax=search.plot(x='K',y=['train_error','validation_error'],marker='o',logx=True,figsize=(7,3.5),ylim=(0,0.5))
ax.set_ylabel('Misclassification rate'); ax.set_title('Complexity check: smaller K is more flexible'); plt.tight_layout(); plt.show()


## 4B · Is the test result impressive? (4 minutes)
Before you read the model's score, look at the test set and at a **majority baseline**: a rule that always predicts *no default*.

**Predict before running:** If we predict *no default* for everyone, what accuracy do we get? How many of the actual defaults does that rule find?

**Accuracy:** fraction of all labels predicted correctly. **Default recall:** fraction of recorded defaults found. **Balanced accuracy:** average of recall for default and recall for non-default. If a model always predicts non-default, balanced accuracy is 50% when both classes occur. These are diagnostic metrics, not a business utility calculation.


In [ ]:
def score(m,d,features,y):
    p=m.predict(d[features])
    return {'n':len(d),'accuracy':accuracy_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p),
            'default_recall':recall_score(y,p,zero_division=0)}
dummy=DummyClassifier(strategy='most_frequent').fit(train[FEATURES],train.default_12m)
results=pd.DataFrame({'Majority baseline':score(dummy,test,FEATURES,test.default_12m),
                      'Selected KNN':score(best,test,FEATURES,test.default_12m)}).T
print('Test cases:',len(test),'| did not default:',int((test.default_12m==0).sum()),'| defaulted:',int(test.default_12m.sum()))
display(results.round(3))
print('The test result estimates performance among historically approved applicants, against recorded labels.')


## 5 · Leakage demonstration (3 minutes)
A deliberately invalid model adds `collection_flag`, a record made after the outcome. Run it, then reject it for application-time prediction even if its score looks attractive. A train/test split cannot make a future field available earlier.

Both comparisons use the same rows, K = 5, and preprocessing. This is a demonstration of two predefined feature sets, **not another model-selection round**. Do not choose a deployment model using these test results.

**Answer first:** Could `collection_flag` exist at the moment of application? What should we do if its score looks perfect?


In [ ]:
bad_features=FEATURES+['collection_flag']
leaky=knn(5).fit(train[bad_features],train.default_12m)
clean5=knn(5).fit(train[FEATURES],train.default_12m)
leak_results=pd.DataFrame({'K=5, application features':score(clean5,test,FEATURES,test.default_12m),
                           'K=5, future field (INVALID)':score(leaky,test,bad_features,test.default_12m)}).T
display(leak_results.round(3))


## 6 · Human audit of selection and deployment (5 minutes)
The **application register** includes rejected applicants. It supports comparisons of observable characteristics, but their default labels are missing. Missing default means **unobserved**, not successful repayment.

Compare the register, recorded loans, and next-quarter applications. First-time customers have zero representation in the modeling file by construction. A random split of that file preserves this absence.

**Answer before revealing outcomes:** What can you establish from these counts? What can you not establish? Is removing `segment` from predictors enough to create evidence for first-time customers?


In [ ]:
population_summary=pd.DataFrame([
 {'population':name,'n':len(d),'new_share':d.segment.eq('new').mean(),'mean_credit':d.credit_score.mean(),
  'label_observed_share':d.default_12m.notna().mean()}
 for name,d in [('Historical register',applications),('Recorded loans',clean),('Future applications',future)]])
display(population_summary.round(3))
display(pd.crosstab(applications.segment,applications.approved,normalize='index').round(3))


## 7 · Simulation-only reveal (3 minutes)
The next cell opens the **answer vault**. It exposes the potential default outcome under the fixed loan terms for every simulated person. This is impossible to do merely by cleaning a real rejected-applicant file.

It evaluates one frozen model. It does not train, tune, or choose features using vault outcomes. The change in future performance combines changes in population mix, feature coverage, and label measurement. It is not a clean causal estimate of any single bias.

Compare the **same historical test predictions** against recorded and oracle labels to isolate the evaluation-label difference. Then compare future subgroups. An underrepresented group can be risky to deploy to even if aggregate accuracy remains high.


In [ ]:
hist_oracle=historical_vault.loc[test.applicant_id].to_numpy()
future_clean=schema_clean(future)
reveal_rows=[]
for name,d,y in [('Historical test / recorded',test,test.default_12m),
                 ('Historical test / oracle',test,hist_oracle),
                 ('Future all / oracle',future_clean,future_vault.loc[future_clean.applicant_id].to_numpy())]:
    reveal_rows.append({'evaluation':name,**score(best,d,FEATURES,y)})
for segment in ['established','new']:
    d=future_clean.loc[future_clean.segment.eq(segment)]
    reveal_rows.append({'evaluation':'Future '+segment+' / oracle',**score(best,d,FEATURES,future_vault.loc[d.applicant_id].to_numpy())})
reveal=pd.DataFrame(reveal_rows); display(reveal.round(3))
print('Historical test label disagreement with oracle:',round(np.mean(test.default_12m.to_numpy()!=hist_oracle),3))
print('Disagreement is measurable here ONLY because this is a simulation.')


## 8 · Your decision readiness audit
Fill the table in your own copy. Use one number, field definition, or policy fact for each concern. Separate what is known from what would need investigation.

| Dimension | Evidence | Decision consequence | Response (Fix, Learn, or Limit) and next action |
|---|---|---|---|
| Representativeness | | | |
| Selection mechanism | | | |
| Leakage | | | |
| Label quality | | | |
| Deployment mismatch | | | |

Write about 120 words: **Deploy / Do not deploy / Deploy with conditions**. Include four things: (1) your recommendation and its scope: for whom can the model be used? (2) your evidence: one model result and two data facts; (3) the evidence gap: what this file cannot tell you; (4) your next action and learning plan: who owns it, and how you will obtain the missing evidence. A condition such as “monitor carefully” needs a metric, owner, and action trigger. Fix means change the process, Learn means generate better evidence, Limit means narrow the decision. Any pilot idea is illustrative and would need legal and compliance review.

**Optional sensitivity:** change only `APPROVAL_CUTOFF`, `FUTURE_NEW_SHARE`, or `LABEL_MISS_RATE`; rerun. State your prediction first. Save both runs. Do not treat the second test result as a new untouched test. A confirmatory claim requires a fresh evaluation cohort.


## 9 · Optional real-data transfer (90-minute route or homework preparation)
**UCI Bank Marketing**: Moro, S., Rita, P., & Cortez, P. (2014), DOI https://doi.org/10.24432/C5K306. Dataset license: CC BY 4.0. Source: https://archive.ics.uci.edu/dataset/222/bank+marketing.

The embedded, unmodified `bank-additional.csv` contains 4,119 rows and 20 inputs plus target. UCI describes it as a random 10% subset of the larger additional dataset, covering a Portuguese bank's telephone campaigns. These are real data, unlike the loan simulation. The CSV is compressed inside this notebook so no download is needed. It was retrieved on 2026-09-20.

**New decision:** whom should a campaign manager call, before the call starts? `duration` measures the last call's duration. Compare empty cells with literal `unknown` categories. A value of `999` for `pdays` means no previous contact in this **additional** version, not 999 ordinary elapsed days. Read the data dictionary before cleaning.

**Audit questions for your report.** Real data do not have an answer vault: you can audit the file and its documentation, but you cannot check hidden outcomes.
1. **Target leakage:** Why is `duration` (call length in seconds) unavailable before the telemarketer dials? What happens to a live campaign if `duration` stays in the model?
2. **Schema trap:** Why does `pdays = 999` distort Euclidean distance if it is passed directly into `StandardScaler()` and KNN? How should an audit-minded data scientist handle this field?

Run the next cell for the numbers. Then answer both questions in your own words and cite the data dictionary.

The notebook does not establish that this dataset represents all potential bank customers. Propose the records you would request to investigate coverage. Do not claim the loan simulation's selection rule applies here.


In [ ]:
import base64, gzip, io, hashlib
_BANK_CSV_GZIP_B64 = '...'
bank_bytes=gzip.decompress(base64.b64decode(_BANK_CSV_GZIP_B64))
assert hashlib.sha256(bank_bytes).hexdigest()=='7e59cf650004d65d1c9d6b08553bad2ee9a9ad70d594f536e3c584ee6ed5df50'
bank=pd.read_csv(io.BytesIO(bank_bytes),sep=';')
print('Shape:',bank.shape)
print('Empty cells:',int(bank.isna().sum().sum()))
print('Literal unknown cells:',int(bank.eq('unknown').sum().sum()))
print('No previous contact (pdays=999):',int(bank.pdays.eq(999).sum()))
display(bank[['duration','pdays','y']].head())
display(bank.y.value_counts(normalize=True).rename('share'))


In [ ]:
# Evidence for the two Section 9 questions (no answer vault here: these are file statistics only)
from sklearn.metrics import roc_auc_score
y_bin = bank['y'].eq('yes').astype(int)
print('Q1  AUC for predicting y from duration alone:', round(roc_auc_score(y_bin, bank['duration']), 3))
p = bank['pdays']
mu, sd = p.mean(), p.std(ddof=0)          # the mean and spread StandardScaler would use
print('Q2  pdays mean / std:', round(mu, 1), '/', round(sd, 1), '| share equal to 999:', round(p.eq(999).mean(), 3))
print('    scaled value of 999:', round((999 - mu) / sd, 2),
      '| scaled 3 days:', round((3 - mu) / sd, 2), '| scaled 20 days:', round((20 - mu) / sd, 2))
print('    yes-rate, never contacted vs previously contacted:', round(y_bin[p.eq(999)].mean(), 3), 'vs', round(y_bin[p.ne(999)].mean(), 3))


## 10 · Save your evidence
The cell below exports the generated results and audit fields. In Colab, use the Files panel to download `audit_evidence.json`, then save the notebook with its outputs. No files are sent anywhere automatically. A full assignment also needs the memo, citations, and AI-use disclosure.

**Sources for methods:** scikit-learn, [Common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html) and [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html). Selective-label motivation: Dennis Wei (2021), [ICML paper](https://proceedings.mlr.press/v139/wei21a.html). All simulation rules and classroom exercises are original teaching examples, not empirical claims about a lender.


In [ ]:
evidence={'controls':{'seed':SEED,'cutoff':APPROVAL_CUTOFF,'future_new_share':FUTURE_NEW_SHARE,'label_miss_rate':LABEL_MISS_RATE},
 'selected_k':best_k,'search':search.to_dict(orient='records'),'test':results.reset_index().to_dict(orient='records'),
 'leakage':leak_results.reset_index().to_dict(orient='records'),'populations':population_summary.to_dict(orient='records'),
 'simulation_only_reveal':reveal.to_dict(orient='records'),'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__}}
with open('audit_evidence.json','w') as f: json.dump(evidence,f,indent=2)
print('Saved audit_evidence.json')
